In [1]:
import pandas as pd
import numpy as np
import pathlib
from tqdm.auto import tqdm

import hydra
from omegaconf import DictConfig, OmegaConf

import torch
from torch_geometric import seed_everything

import ray

In [2]:
node = !hostname
if "sc" in node[0]:
    base_path = "/sc-projects/sc-proj-ukb-cvd"
else: 
    base_path = "/data/analysis/ag-reils/ag-reils-shared/cardioRS"
print(base_path)

project_label = "22_medical_records"
project_path = f"{base_path}/results/projects/{project_label}"
figure_path = f"{project_path}/figures"
output_path = f"{project_path}/data"

pathlib.Path(figure_path).mkdir(parents=True, exist_ok=True)
pathlib.Path(output_path).mkdir(parents=True, exist_ok=True)

In [3]:
records = pd.read_feather(f"{output_path}/baseline_records_220223.feather")

In [4]:
records

            eid  OMOP_1000560  OMOP_1000632  OMOP_1000772  OMOP_1000979  \
0       1475839           0.0           0.0           0.0           0.0   
1       1475840           0.0           0.0           0.0           0.0   
2       1475853           0.0           0.0           0.0           0.0   
3       1475862           0.0           1.0           0.0           0.0   
4       1475871           0.0           0.0           0.0           0.0   
...         ...           ...           ...           ...           ...   
502455  6025150           0.0           0.0           0.0           0.0   
502456  6025165           0.0           0.0           0.0           0.0   
502457  6025173           0.0           0.0           0.0           0.0   
502458  6025182           0.0           0.0           0.0           0.0   
502459  6025198           0.0           0.0           0.0           0.0   

        OMOP_1000995  OMOP_1001419  OMOP_1036059  OMOP_1036157  OMOP_1036228  \
0                0.

In [5]:
records = records.set_index("eid")

In [6]:
records.ge(1).astype(int)

         OMOP_1000560  OMOP_1000632  OMOP_1000772  OMOP_1000979  OMOP_1000995  \
eid                                                                             
1475839             0             0             0             0             0   
1475840             0             0             0             0             0   
1475853             0             0             0             0             0   
1475862             0             1             0             0             0   
1475871             0             0             0             0             0   
...               ...           ...           ...           ...           ...   
6025150             0             0             0             0             0   
6025165             0             0             0             0             0   
6025173             0             0             0             0             0   
6025182             0             0             0             0             0   
6025198             0       

In [7]:
records = pd.read_feather(f"{output_path}/baseline_records_220223.feather").set_index("eid").ge(1).astype(int)

In [8]:
records

         OMOP_1000560  OMOP_1000632  OMOP_1000772  OMOP_1000979  OMOP_1000995  \
eid                                                                             
1475839             0             0             0             0             0   
1475840             0             0             0             0             0   
1475853             0             0             0             0             0   
1475862             0             1             0             0             0   
1475871             0             0             0             0             0   
...               ...           ...           ...           ...           ...   
6025150             0             0             0             0             0   
6025165             0             0             0             0             0   
6025173             0             0             0             0             0   
6025182             0             0             0             0             0   
6025198             0       

In [9]:
records.sum(axis=1)

eid
1475839    146
1475840    129
1475853     47
1475862    128
1475871     63
          ... 
6025150      3
6025165     14
6025173     19
6025182     20
6025198     25
Length: 502460, dtype: int64

In [10]:
records_freq = records.sum(axis=0)

In [11]:
records_freq

OMOP_1000560      113
OMOP_1000632    32909
OMOP_1000772       33
OMOP_1000979      177
OMOP_1000995     6257
                ...  
OMOP_997276      6892
OMOP_997496        11
OMOP_997881      1813
OMOP_998394      5315
OMOP_998415     13320
Length: 16201, dtype: int64

In [12]:
records_freq = records.sum(axis=0).sort_values()

In [13]:
records_freq

OMOP_4064648          0
OMOP_435135           0
OMOP_44805472         0
OMOP_44805473         0
OMOP_19089810         0
                  ...  
OMOP_320128       69817
OMOP_44793690     74262
OMOP_4214956      83117
OMOP_1713332      93887
OMOP_4301351     129430
Length: 16201, dtype: int64

In [14]:
records_freq = records.sum(axis=0).sort_values(ascending=False)

In [15]:
records_freq

OMOP_4301351     129430
OMOP_1713332      93887
OMOP_4214956      83117
OMOP_44793690     74262
OMOP_320128       69817
                  ...  
OMOP_434499           0
OMOP_44793138         0
OMOP_4345207          0
OMOP_198260           0
OMOP_4146291          0
Length: 16201, dtype: int64

In [16]:
records_freq.to_frame()

                    0
OMOP_4301351   129430
OMOP_1713332    93887
OMOP_4214956    83117
OMOP_44793690   74262
OMOP_320128     69817
...               ...
OMOP_434499         0
OMOP_44793138       0
OMOP_4345207        0
OMOP_198260         0
OMOP_4146291        0

[16201 rows x 1 columns]

In [17]:
concepts_raw = pd.read_feather("/sc-projects/sc-proj-ukb-cvd/data/mapping/athena/CONCEPT.feather")

In [18]:
concepts_raw

         concept_id                           concept_name domain_id  \
0          45756805                   Pediatric Cardiology  Provider   
1          45756804               Pediatric Anesthesiology  Provider   
2          45756803  Pathology-Anatomic/Pathology-Clinical  Provider   
3          45756802                  Pathology - Pediatric  Provider   
4          45756801          Pathology - Molecular Genetic  Provider   
...             ...                                    ...       ...   
8179125    35895436                               Oxyargin      Drug   
8179126    35895825                                Rolufta      Drug   
8179127    35894932                                M-Trate      Drug   
8179128    35895797                               Ritalmex      Drug   
8179129    35895377                                 Neuro3      Drug   

        vocabulary_id     concept_class_id standard_concept concept_code  \
0                ABMS  Physician Specialty                S

In [19]:
concepts_raw["records"] = "OMOP_" + concepts_raw["concept_id"]

In [20]:
concepts_raw["records"] = "OMOP_" + concepts_raw["concept_id"].astype(str)

In [21]:
concepts_raw = pd.read_feather("/sc-projects/sc-proj-ukb-cvd/data/mapping/athena/CONCEPT.feather")
concepts_raw["record"] = "OMOP_" + concepts_raw["concept_id"].astype(str)
concept_raw = concepts_raw.set_index("record")

In [22]:
records_freq = records.sum(axis=0).sort_values(ascending=False).to_frame().reset_index()
records_freq

               index       0
0       OMOP_4301351  129430
1       OMOP_1713332   93887
2       OMOP_4214956   83117
3      OMOP_44793690   74262
4        OMOP_320128   69817
...              ...     ...
16196    OMOP_434499       0
16197  OMOP_44793138       0
16198   OMOP_4345207       0
16199    OMOP_198260       0
16200   OMOP_4146291       0

[16201 rows x 2 columns]

In [23]:
records_freq = records.sum(axis=0).sort_values(ascending=False).to_frame().reset_index()
records_freq.columns = ["record", "n"]
records_freq = records_freq.set_index("record")
records_freq

                    n
record               
OMOP_4301351   129430
OMOP_1713332    93887
OMOP_4214956    83117
OMOP_44793690   74262
OMOP_320128     69817
...               ...
OMOP_434499         0
OMOP_44793138       0
OMOP_4345207        0
OMOP_198260         0
OMOP_4146291        0

[16201 rows x 1 columns]

In [24]:
concept_raw

               concept_id                           concept_name domain_id  \
record                                                                       
OMOP_45756805    45756805                   Pediatric Cardiology  Provider   
OMOP_45756804    45756804               Pediatric Anesthesiology  Provider   
OMOP_45756803    45756803  Pathology-Anatomic/Pathology-Clinical  Provider   
OMOP_45756802    45756802                  Pathology - Pediatric  Provider   
OMOP_45756801    45756801          Pathology - Molecular Genetic  Provider   
...                   ...                                    ...       ...   
OMOP_35895436    35895436                               Oxyargin      Drug   
OMOP_35895825    35895825                                Rolufta      Drug   
OMOP_35894932    35894932                                M-Trate      Drug   
OMOP_35895797    35895797                               Ritalmex      Drug   
OMOP_35895377    35895377                                 Neuro3

In [25]:
records_freq.merge(concept_raw, left_index=True, right_index=True, how="left")

                    n  concept_id  \
record                              
OMOP_4301351   129430     4301351   
OMOP_1713332    93887     1713332   
OMOP_4214956    83117     4214956   
OMOP_44793690   74262    44793690   
OMOP_320128     69817      320128   
...               ...         ...   
OMOP_434499         0      434499   
OMOP_44793138       0    44793138   
OMOP_4345207        0     4345207   
OMOP_198260         0      198260   
OMOP_4146291        0     4146291   

                                                    concept_name    domain_id  \
record                                                                          
OMOP_4301351                                  Surgical procedure    Procedure   
OMOP_1713332                                         amoxicillin         Drug   
OMOP_4214956              History of clinical finding in subject  Observation   
OMOP_44793690          Radiology of one body area or <20 minutes    Procedure   
OMOP_320128                     

In [26]:
records_freq_md = records_freq.merge(concept_raw, left_index=True, right_index=True, how="left")

In [27]:
records_freq_md

                    n  concept_id  \
record                              
OMOP_4301351   129430     4301351   
OMOP_1713332    93887     1713332   
OMOP_4214956    83117     4214956   
OMOP_44793690   74262    44793690   
OMOP_320128     69817      320128   
...               ...         ...   
OMOP_434499         0      434499   
OMOP_44793138       0    44793138   
OMOP_4345207        0     4345207   
OMOP_198260         0      198260   
OMOP_4146291        0     4146291   

                                                    concept_name    domain_id  \
record                                                                          
OMOP_4301351                                  Surgical procedure    Procedure   
OMOP_1713332                                         amoxicillin         Drug   
OMOP_4214956              History of clinical finding in subject  Observation   
OMOP_44793690          Radiology of one body area or <20 minutes    Procedure   
OMOP_320128                     

In [28]:
artifact_path = "/sc-projects/sc-proj-ukb-cvd/data/2_datasets_pre/211110_anewbeginning/artifacts/record_frequencies.p"

In [29]:
artifact_path = "/sc-projects/sc-proj-ukb-cvd/data/2_datasets_pre/211110_anewbeginning/artifacts/record_frequencies.p"

In [30]:
records_freq_md.reset_index().to_feather(artifact_path)

In [31]:
run = wandb.init(project="RecordGraphs", entity="cardiors", tags=["artifacts"])

with open(artifact_path, "wb") as fh:
    cctx = zstandard.ZstdCompressor()
    with cctx.stream_writer(fh) as compressor:
        compressor.write(pickle.dumps(data, protocol=pickle.HIGHEST_PROTOCOL))

artifact = wandb.Artifact("RecordFrequencies", type="prepare_records")
artifact.add_reference(f"file://{artifact_path}", "RecordsMetadata", checksum=True)
run.log_artifact(artifact)

run.finish()

In [32]:
import wandb
run = wandb.init(project="RecordGraphs", entity="cardiors", tags=["artifacts"])

with open(artifact_path, "wb") as fh:
    cctx = zstandard.ZstdCompressor()
    with cctx.stream_writer(fh) as compressor:
        compressor.write(pickle.dumps(data, protocol=pickle.HIGHEST_PROTOCOL))

artifact = wandb.Artifact("RecordFrequencies", type="prepare_records")
artifact.add_reference(f"file://{artifact_path}", "RecordsMetadata", checksum=True)
run.log_artifact(artifact)

run.finish()

In [33]:
import wandb
import pickle
import zstandard 

run = wandb.init(project="RecordGraphs", entity="cardiors", tags=["artifacts"])

with open(artifact_path, "wb") as fh:
    cctx = zstandard.ZstdCompressor()
    with cctx.stream_writer(fh) as compressor:
        compressor.write(pickle.dumps(data, protocol=pickle.HIGHEST_PROTOCOL))

artifact = wandb.Artifact("RecordFrequencies", type="prepare_records")
artifact.add_reference(f"file://{artifact_path}", "RecordsMetadata", checksum=True)
run.log_artifact(artifact)

run.finish()